# 09 — Isolation Forest anomaly experiment

The detector was fitted on legitimate (`Class = 0`) rows from the training partition only. A higher `-score_samples` value means more anomalous; it is not a fraud probability. The supervised and anomaly rankings remain separate, and their top-K overlap is descriptive.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
results = json.loads((ROOT / "reports/ml/latest_results.json").read_text(encoding="utf-8"))
anomaly = results["anomaly"]
print("Fit protocol:", anomaly["fit"]["training_label_filter"])
print("Score definition:", anomaly["fit"]["score_definition"])
summary = pd.DataFrame.from_dict(
    {"Validation": anomaly["validation"], "Final test": anomaly["final_test"]},
    orient="index",
)[
    [
        "rows",
        "fraud_count",
        "average_precision",
        "roc_auc",
        "score_direction",
        "score_is_probability",
    ]
]
display(summary)
final_budgets = anomaly["final_test"]["alert_budgets"]
budget_table = pd.DataFrame(
    {
        "Budget": [row["budget_percentage"] for row in final_budgets],
        "K": [row["k"] for row in final_budgets],
        "Fraud captured": [row["true_positives"] for row in final_budgets],
        "Recall@K": [row["recall_at_k"] for row in final_budgets],
    }
)
display(budget_table)
display(Image(filename=str(ROOT / "reports/ml/figures/anomaly_score_distribution.png")))
display(Image(filename=str(ROOT / "reports/ml/figures/alert_budget_comparison.png")))